In [1]:
!pip install bioc faiss-gpu-cu12
!pip uninstall -y faiss-cpu faiss-gpu-cu12
!pip install faiss-gpu-cu12
!pip install spacy

  Using cached bioc-2.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached faiss_gpu_cu12-1.14.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached jsonlines-4.0.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached intervaltree-3.2.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached docopt-0.6.2-py2.py3-none-any.whl
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
Using cached bioc-2.1-py3-none-any.whl (33 kB)
Using cached faiss_gpu_cu12-1.14.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (48.4 MB)
Using cached jsonlines-4.0.0-py3-none-any.whl (8.7 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
Using cached intervaltree-3.2.1-py2.py3-none-any.whl (25 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Attempting uninstall: 

In [2]:
from pathlib import Path

DATA_DIR = Path('')

In [3]:
from bioc import biocxml
import gzip

def load_bioc(path):
    with gzip.open(path, "rt", encoding="utf-8") as fp:
        collection = biocxml.load(fp)
    return collection.documents

In [4]:
def load_medmentions():
  DATA_DIR = '../Data/'
  TRAIN_FILE = DATA_DIR + "MedMentions/medmentions_st21pv_train.bioc.xml.gz"
  VAL_FILE = DATA_DIR + "MedMentions/medmentions_st21pv_val.bioc.xml.gz"
  TEST_FILE = DATA_DIR + "MedMentions/medmentions_st21pv_test.bioc.xml.gz"

  train_docs = load_bioc(TRAIN_FILE)
  val_docs = load_bioc(VAL_FILE)
  test_docs = load_bioc(TEST_FILE)

  print(f"{len(train_docs)} training documents")
  print(f"{len(val_docs)} validation documents")
  print(f"{len(test_docs)} test documents")

  return train_docs, val_docs, test_docs

In [5]:
train_docs, val_docs, test_docs = load_medmentions()

2635 training documents
878 validation documents
879 test documents


In [6]:
def get_UMLS_files():
  UMLS_FOLDER = "../Data/Ontology/"

  MRCONSO_FILE = UMLS_FOLDER + "MRCONSO.RRF"
  MRDEF_FILE = UMLS_FOLDER + "MRDEF.RRF"
  MRSTY_FILE = UMLS_FOLDER + "MRSTY.RRF"

  return MRCONSO_FILE, MRDEF_FILE, MRSTY_FILE

In [7]:
# UMLS has vastly more CUIS than medmentions uses, so to save time, we only need to consider the ones it uses instead of them all
def get_medmentions_cuis():
  used_cuis = set()

  for doc in train_docs + val_docs + test_docs:
      for passage in doc.passages:
          for anno in passage.annotations:
              cui = anno.infons.get("concept_id")
              if cui:
                  used_cuis.add(cui)

  return used_cuis

In [8]:
used_cuis = get_medmentions_cuis()

In [39]:
from collections import defaultdict
from tqdm import tqdm

def prep_UMLS_concepts(MRCONSO_FILE):
  concepts = defaultdict(lambda: {
    "name": None,
    "aliases": set(),
    "name_score": None
  })

  with open(MRCONSO_FILE, encoding="utf-8") as f:
      for line in tqdm(f, desc="Loading MRCONSO"):
          fields = line.rstrip("\n").split("|")

          cui = fields[0]
          lang = fields[1]
          term_status = fields[2]
          string_type = fields[4]
          is_preferred = fields[6]
          term_type = fields[12]
          term = fields[14]

          if lang != "ENG":
              continue
          if f"UMLS:{cui}" not in used_cuis:
            continue
          if not term.strip():
              continue

          concepts[cui]["aliases"].add(term)

          score = (
                int(term_status == "P"),
                int(string_type == "PF"),
                int(is_preferred == "Y"),
                int(term_type in {"PT", "PN"}),
                -len(term)
            )
          
          current_score = concepts[cui]["name_score"]

          if current_score is None or score > current_score:
            concepts[cui]["name"] = term
            concepts[cui]["name_score"] = score
              
  return concepts

In [40]:
def prep_UMLS_sty(MRSTY_FILE):
    semantic_types = {}

    with open(MRSTY_FILE, encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading MRSTY"):
            fields = line.rstrip("\n").split("|")

            cui = fields[0]
            sty = fields[3]

            # if cui in used_cuis:
            #     continue
            if cui in semantic_types:
                semantic_types[cui].append(sty)
            else:
                semantic_types[cui] = [sty]
    return semantic_types

In [41]:
def generate_ontology():
  MRCONSO_FILE, MRDEF_FILE, MRSTY_FILE = get_UMLS_files()
  concepts = prep_UMLS_concepts(MRCONSO_FILE)
  semantic_types = prep_UMLS_sty(MRSTY_FILE)

  ontology = []

  for cui, concept in concepts.items():
      ontology.append({
          "id": f"UMLS:{cui}",
          "name": concept["name"],
          "aliases": sorted(concept["aliases"]),
          # "definition": definitions.get(cui, ""),
          "types": semantic_types.get(cui, [])
      })
  return ontology

In [42]:
ontology = generate_ontology()

Loading MRCONSO: 18064970it [00:36, 500378.89it/s]
Loading MRSTY: 3876927it [00:07, 528953.18it/s] 


In [13]:
def cleanse_anno(anno):
    anno = (
      (anno).replace("α", "alpha")
      .replace("β", "beta")
      .replace("γ", "gamma")
      .replace("δ", "delta")
      .replace("κ", "kappa")
      .replace("λ", "lambda")
      .replace("μ", "mu")
      .replace("ω", "omega")
      .replace(" ii ", " 2 ")
      .replace("(ii)", "(2)")
      .replace("ii ", "2 ", 1)
      .replace(" iii ", " 3 ")
      .replace("(iii)", "(3)")
      .replace("iii ", "3 ", 1)
              )
  
    if anno.endswith(" ii"):
      anno = " 2".join(anno.text.rsplit(" ii", 1))
    if anno.endswith(" iii"):
      anno = " 3".join(anno.text.rsplit(" iii", 1))

    return anno

In [14]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

model = SentenceTransformer(MODEL_NAME)

No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


In [15]:
def make_entity_texts(train_docs, val_docs, test_docs):
    entity_texts = []
    
    for doc in train_docs+val_docs+test_docs:
      for passage in doc.passages:
        for anno in passage.annotations:
          anno.text = cleanse_anno(anno.text)
          entity_texts.append(anno.text)
    
    return sorted(set(entity_texts)) 

In [16]:
entity_texts = make_entity_texts(train_docs, val_docs, test_docs)

In [17]:
CANDIDATES_TO_GENERATE = 5

In [18]:
SAPBERT_MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

In [19]:
import json
import faiss
import torch

CACHE_DIRECTORY = Path("../Cache/SapBERT")
EMBEDDINGS_PATH = CACHE_DIRECTORY / "umls_embeddings.npy"
ONTOLOGY_PATH = CACHE_DIRECTORY / "umls_ontology.json"


def get_sapbert_model(model_name=SAPBERT_MODEL_NAME):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"loading sapbert on {device}")

    return SentenceTransformer(model_name, device=device)


def encode_entities(entity_names, model, batch_size=32):
    embeddings = model.encode(
        entity_names,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.array(embeddings)


def rank_entities(entity_vectors, index, candidates_to_generate):
    return index.search(entity_vectors, k=candidates_to_generate)


def generate_sapbert_scores_indices(entity_texts, index, model, candidates_to_generate=CANDIDATES_TO_GENERATE):
   cleaned_entity_texts = [cleanse_anno(text) for text in entity_texts]
   
   entity_vectors = encode_entities(cleaned_entity_texts, model)

   return rank_entities(entity_vectors, index, candidates_to_generate) # scores, indices


def load_sapbert_index():
    ontology_vectors = np.load(EMBEDDINGS_PATH)

    return create_faiss_index(ontology_vectors)


def  build_sapbert_index(ontology, model):
    entity_names = [entity["name"] for entity in ontology]

    ontology_vectors = encode_entities(entity_names, model)

    return create_faiss_index(ontology_vectors)


def sapbert_cache_is_compatible(ontology, ontology_path=ONTOLOGY_PATH):
    if not ontology_path.exists():
        return False

    try:
        with open(ontology_path, "r", encoding="utf-8") as file:
            cached_ontology = json.load(file)
    except (OSError, json.JSONDecodeError):
        return False

    return cached_ontology == ontology


def get_or_build_sapbert_index(ontology, model):
    cache_exists = (
        EMBEDDINGS_PATH.exists()
        and ONTOLOGY_PATH.exists()
    )

    if cache_exists and sapbert_cache_is_compatible(ontology):
        return load_sapbert_index()

    if cache_exists:
        print(
            "Cached SapBERT embeddings are incompatible "
            "with the current ontology."
        )
    else:
        print("No cached SapBERT ontology index found.")

    print("Encoding ontology for the first time...")

    return build_and_save_sapbert_index(ontology, model)


def build_and_save_sapbert_index(
    ontology,
    model,
    embeddings_path=EMBEDDINGS_PATH,
    ontology_path=ONTOLOGY_PATH
):
    CACHE_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True
    )

    entity_names = [entity["name"] for entity in ontology]

    ontology_vectors = encode_entities(entity_names, model, batch_size=32)

    ontology_vectors = np.asarray(ontology_vectors, dtype=np.float32)

    np.save(embeddings_path, ontology_vectors)

    index = create_faiss_index(ontology_vectors)

    with open(ontology_path, "w", encoding="utf-8") as file:
        json.dump(
            ontology,
            file,
            ensure_ascii=False
        )

    print(
        f"Saved {len(ontology_vectors):,} ontology embeddings "
        f"to {embeddings_path}"
    )

    return index


def create_faiss_index(ontology_vectors):
    ontology_vectors = np.asarray(ontology_vectors, dtype=np.float32)

    ontology_vectors = np.ascontiguousarray(ontology_vectors)

    dimension = ontology_vectors.shape[1]

    cpu_index = faiss.IndexFlatIP(dimension)
    cpu_index.add(ontology_vectors)

    if torch.cuda.is_available():
        resources = faiss.StandardGpuResources()

        gpu_index = faiss.index_cpu_to_gpu(
            resources,
            0,
            cpu_index
        )

        return gpu_index

    return cpu_index

In [ ]:
sapbert_index = get_or_build_sapbert_index(ontology, model)

In [ ]:
sapbert_scores, sapbert_indices = generate_sapbert_scores_indices(entity_texts, sapbert_index, model)

In [ ]:
candidate_lookup = { entity_text:sapbert_indices[i].tolist() for i,entity_text in enumerate(entity_texts) }
scores_lookup = { entity_text:sapbert_scores[i].tolist() for i,entity_text in enumerate(entity_texts) }

In [20]:
import spacy

nlp = spacy.blank("en")
punct_chars = ['!', '.', '?', '։', '؟', '۔', '܀', '܁', '܂', '߹', '।', '॥', '၊', '။', '።',
                 '፧', '፨', '᙮', '᜵', '᜶', '᠃', '᠉', '᥄', '᥅', '᪨', '᪩', '᪪', '᪫',
                 '᭚', '᭛', '᭞', '᭟', '᰻', '᰼', '᱾', '᱿', '‼', '‽', '⁇', '⁈', '⁉',
                 '⸮', '⸼', '꓿', '꘎', '꘏', '꛳', '꛷', '꡶', '꡷', '꣎', '꣏', '꤯', '꧈',
                 '꧉', '꩝', '꩞', '꩟', '꫰', '꫱', '꯫', '﹒', '﹖', '﹗', '！', '．', '？',
                 '𐩖', '𐩗', '𑁇', '𑁈', '𑂾', '𑂿', '𑃀', '𑃁', '𑅁', '𑅂', '𑅃', '𑇅',
                 '𑇆', '𑇍', '𑇞', '𑇟', '𑈸', '𑈹', '𑈻', '𑈼', '𑊩', '𑑋', '𑑌', '𑗂',
                 '𑗃', '𑗉', '𑗊', '𑗋', '𑗌', '𑗍', '𑗎', '𑗏', '𑗐', '𑗑', '𑗒', '𑗓',
                 '𑗔', '𑗕', '𑗖', '𑗗', '𑙁', '𑙂', '𑜼', '𑜽', '𑜾', '𑩂', '𑩃', '𑪛',
                 '𑪜', '𑱁', '𑱂', '𖩮', '𖩯', '𖫵', '𖬷', '𖬸', '𖭄', '𛲟', '𝪈', '｡', '。', '\n']
nlp.add_pipe("sentencizer", config={"punct_chars": punct_chars})
nlp.max_length = 5000000 # random high enough number to not cause issues

In [21]:
from types import SimpleNamespace
import random
from transformers import AutoTokenizer
import numpy as np

TOKEN_LIMIT = 512

def prep_tokenizer(
      model_name="microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
      ):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tokenizer.add_tokens(['[ENTITY]', '[E]', '[/E]', '[TYPES]', '[SEP]', '[SCORE]', ],)

    return tokenizer

def split_passage_into_sentences(passage, nlp):
    doc = nlp(passage.text)

    sentence_passages = []

    for sent in doc.sents:
        sent_start = passage.offset + sent.start_char
        sent_end = passage.offset + sent.end_char

        sent_annos = [anno for anno in passage.annotations
                    if anno.locations[0].offset >= sent_start
                    and anno.locations[0].offset + anno.locations[0].length <= sent_end]


        sentence_passages.append(SimpleNamespace(
            text=sent.text,
            offset=sent_start,
            annotations=sent_annos
        ))

    return sentence_passages

def get_token_count(text, tokenizer):
    return len(
        tokenizer(
            text,
            truncation=False
        )["input_ids"]
    )

def get_anno_label(anno, identifier):
    gold_id = anno.infons.get("concept_id")

    if gold_id is None:
        label = None
    
    else:
        candidate_is_correct = (identifier == gold_id)
        label = ("CORRECT" if candidate_is_correct else "INCORRECT")

    return label

def create_candidates(text, candidate_idxs, candidate_scores, ontology, anno):
    candidates = []
    for idx, score in zip(candidate_idxs, candidate_scores):
            if idx == -1:
                continue
            
            identifier = ontology[idx]['id']
            entity_name = ontology[idx]['name']
            entity_types = "; ".join(ontology[idx]["types"])
            entity_score = min(100, int(score * 100))
            start = len(text)
            end = start + len('[ENTITY]')
            text += f"[ENTITY]{entity_name}[TYPES]{entity_types}[SCORE]{entity_score}"
            label = get_anno_label(anno, identifier)

            candidates.append( {'id':identifier, 'name':entity_name, 'start':start, 'end':end, 'label':label } )
    return text, candidates

def build_annotated_sentence(passage, prev_passage, annos):
    annos = sorted(
        annos,
        key=lambda anno: anno.locations[0].offset
    )

    text = prev_passage + "[SEP]"
    previous_end = 0

    for anno in annos:
        anno_start = anno.locations[0].offset - passage.offset
        anno_end = anno_start + anno.locations[0].length
        text += passage.text[previous_end:anno_start]
        text += f"[E]{passage.text[anno_start:anno_end]}[/E]"

        previous_end = anno_end

    text += passage.text[previous_end:]

    return text

def build_chunk(passage, prev_passage, annos, candidates_by_anno, scores_by_anno, ontology):
    text = build_annotated_sentence(passage, prev_passage, annos)
    candidates = []

    for anno, candidate_idxs, candidate_scores in zip(annos, candidates_by_anno, scores_by_anno):
        text, candidate = create_candidates(text, candidate_idxs, candidate_scores, ontology, anno)
        candidates.extend(candidate)

    return {'text':text, 'candidates':candidates, 'sentence':passage, 'annotations':annos}

def make_candidates_gold(anno, anno_candidate_idxs, anno_candidate_scores, ontology_id_to_idx):
    
    gold_idx = ontology_id_to_idx.get(
            anno.infons["concept_id"]
        )

    if (
        gold_idx is not None
        and gold_idx not in anno_candidate_idxs
    ):
        anno_candidate_idxs[-1] = gold_idx


    paired = list(zip(anno_candidate_idxs, anno_candidate_scores))
    random.shuffle(paired)

    anno_candidate_idxs, anno_candidate_scores = map(
        list,
        zip(*paired)
    )

    return anno_candidate_idxs, anno_candidate_scores


def fast_anno_with_candidates(passage, prev_passage, annos, candidate_lookup, scores_lookup, ontology_id_to_idx, ontology, force_gold):
    annos = sorted(
        annos,
        key=lambda anno: anno.locations[0].offset
    )

    candidate_idxs = [
        candidate_lookup[anno.text]
        for anno in annos
    ]

    candidate_scores = [
        np.asarray(scores_lookup[anno.text], dtype=np.float32)
        for anno in annos
    ]

    candidate_scores = [
        scores / scores.max() if scores.max() > 0 else scores
        for scores in candidate_scores
    ]

    if force_gold:
        for i, anno in enumerate(annos):
            candidate_idxs[i], candidate_scores[i] = make_candidates_gold(anno, 
                                                                        candidate_idxs[i], 
                                                                        candidate_scores[i], 
                                                                        ontology_id_to_idx)

    text_with_entities = build_annotated_sentence(passage, prev_passage, annos)

    candidates = []
    for anno, indices, score in zip(annos, candidate_idxs, candidate_scores):
        text_with_entities, candidate = create_candidates(text_with_entities, indices, score, ontology, anno)
        candidates.extend(candidate)
        
    return {'text':text_with_entities, 'candidates':candidates, 'sentence':passage, 'annotations':annos}

def slow_anno_with_candidates(passage, prev_passage, annos, candidate_lookup, scores_lookup, ontology_id_to_idx, force_gold, tokenizer, ontology):
    # sorted on where they occur in passage
    annos = sorted(
        annos,
        key=lambda anno: anno.locations[0].offset
    )

    all_chunks = []

    current_annos = []
    current_candidate_idxs = []
    current_candidate_scores = []


    for anno in annos:
        anno_candidate_idxs = candidate_lookup[anno.text]

        anno_candidate_scores = np.asarray(scores_lookup[anno.text], dtype=np.float32)

        anno_candidate_scores = (anno_candidate_scores / anno_candidate_scores.max()
                                if anno_candidate_scores.max() > 0 
                                else anno_candidate_scores)

        if force_gold:
            anno_candidate_idxs, anno_candidate_scores = make_candidates_gold(anno, 
                                                                            anno_candidate_idxs, 
                                                                            anno_candidate_scores, 
                                                                            ontology_id_to_idx)

        proposed_annos = current_annos + [anno]
        proposed_candidate_idxs = (current_candidate_idxs + [anno_candidate_idxs])
        proposed_candidate_scores = (current_candidate_scores + [anno_candidate_scores])

        proposed_chunk = build_chunk(passage, prev_passage, proposed_annos, proposed_candidate_idxs, proposed_candidate_scores, ontology)

        if get_token_count(proposed_chunk["text"], tokenizer) <= TOKEN_LIMIT:
            current_annos = proposed_annos
            current_candidate_idxs = proposed_candidate_idxs
            current_candidate_scores = proposed_candidate_scores
            continue

        if current_annos:
            all_chunks.append(build_chunk(passage, prev_passage, current_annos, current_candidate_idxs, current_candidate_scores, ontology))

        current_annos = [anno]
        current_candidate_idxs = [anno_candidate_idxs]
        current_candidate_scores = [anno_candidate_scores]
            
        single_anno_chunk = build_chunk(passage, prev_passage, current_annos, current_candidate_idxs, current_candidate_scores, ontology)

        if get_token_count(single_anno_chunk["text"], tokenizer) > TOKEN_LIMIT:
            for candidate, score in zip(anno_candidate_idxs, anno_candidate_scores):
                single_candidate_chunk = build_chunk(passage, prev_passage, [anno], [[candidate]], [[score]], ontology)
                if get_token_count(single_candidate_chunk["text"], tokenizer) > TOKEN_LIMIT:
                    single_candidate_chunk["text"] = truncate_entity_to_token_limit(single_candidate_chunk["text"], tokenizer)
                all_chunks.append(single_candidate_chunk)

            current_annos = []
            current_candidate_idxs = []
            current_candidate_scores = []
            continue
            
            
        
    if current_annos:
        all_chunks.append(
            build_chunk(
                passage,
                prev_passage,
                current_annos,
                current_candidate_idxs,
                current_candidate_scores,
                ontology
            )
        )

    return all_chunks

def truncate_entity_to_token_limit(text, tokenizer):
    # print(text)
    before_entity, entity_and_after = text.split("[ENTITY]", 1)
    entity_text, after_entity = entity_and_after.rsplit("[TYPES]", 1)

    entity_words = entity_text.split()

    truncated_words = []

    for word in entity_words:
        proposed_words = truncated_words + [word]

        proposed_text = (before_entity
                         + "[ENTITY]"
                         + " ".join(proposed_words)
                         + "[TYPES]"
                         + after_entity
                        )
        if get_token_count(proposed_text, tokenizer) <= TOKEN_LIMIT:
            truncated_words = proposed_words
        else:
            break
    if not truncated_words:
        print(before_entity)
        print(after_entity)
        raise ValueError("token limit exceeded without entity")

    truncated_text = (before_entity
                     + "[ENTITY]"
                     + " ".join(truncated_words)
                     + "[TYPES]"
                     + after_entity
                    )
    
    return truncated_text

    
def make_anno_with_candidates(passage,
    prev_passage,
    annos,
    candidate_lookup,
    scores_lookup,
    ontology,
    ontology_index,
    tokenizer,
    force_gold=False):
    
    candidate_passage = fast_anno_with_candidates(passage, 
                                                  prev_passage, 
                                                  annos, 
                                                  candidate_lookup, 
                                                  scores_lookup, 
                                                  ontology_index, 
                                                  ontology, 
                                                  force_gold)

    if get_token_count(candidate_passage["text"], tokenizer) <= TOKEN_LIMIT:
        return [candidate_passage]
    else:
        return slow_anno_with_candidates(passage, 
                                         prev_passage,
                                         annos, 
                                         candidate_lookup, 
                                         scores_lookup, 
                                         ontology_index, 
                                         force_gold,
                                         tokenizer, 
                                         ontology)

In [22]:
ontology_id_to_idx = {
    concept["id"]: idx
    for idx, concept in enumerate(ontology)
}

In [23]:
tokenizer = prep_tokenizer()

In [24]:
train_sentences = [sentence for doc in train_docs for passage in doc.passages for sentence in split_passage_into_sentences(passage, nlp) if len(sentence.annotations) > 0]
val_sentences = [sentence for doc in val_docs for passage in doc.passages for sentence in split_passage_into_sentences(passage, nlp) if len(sentence.annotations) > 0]
test_sentences = [sentence for doc in test_docs for passage in doc.passages for sentence in split_passage_into_sentences(passage, nlp) if len(sentence.annotations) > 0]

In [ ]:
import random

train_dataset = [
    chunk
    for i, sentence in enumerate(train_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        train_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        candidate_lookup,
        scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
        force_gold=True
    )
]
    
val_dataset = [
    chunk
    for i, sentence in enumerate(val_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        val_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        candidate_lookup,
        scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

test_dataset = [
    chunk
    for i, sentence in enumerate(test_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        test_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        candidate_lookup,
        scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

In [25]:
def validate_evaluation_hits(dataset, k, group_size):
  if k < 1:
    raise Exception("the number of k to evaluate at must be at least 1")

  if k>group_size:
    raise Exception("the number of k to evaluate at must be at most number of candidates")

  if dataset[0] == None:
    raise Exception("The dataset is empty")

  return

In [26]:
def hits_at_k_stage1(dataset, k=5, group_size=5):

    validate_evaluation_hits(dataset, k, group_size)

    hits = []

    for data in dataset:
        candidates = data["candidates"]

        for start in range(0, len(candidates), group_size):
          candidate_group = candidates[start:start+k]

          hit = any(
              candidate["label"] == "CORRECT"
              for candidate in candidate_group
          )

          hits.append(hit)

    return sum(hits) / len(hits)

In [ ]:
print("Stage 1 train Hits@5:", hits_at_k_stage1(train_dataset, k=5))
print("Stage 1 val Hits@5:", hits_at_k_stage1(val_dataset, k=5))
print("Stage 1 test Hits@5:", hits_at_k_stage1(test_dataset, k=5))

In [43]:
NEW_MODEL_NAME = "/mnt/primary/SapBERT_Training/my_sapbert_7"

new_model = SentenceTransformer(NEW_MODEL_NAME)

No sentence-transformers model found with name /mnt/primary/SapBERT_Training/my_sapbert_7. Creating a new one with mean pooling.


In [44]:
new_sapbert_index = build_sapbert_index(ontology, new_model)

Batches:   0%|          | 0/110312 [00:00<?, ?it/s]

In [ ]:
new_sapbert_scores, new_sapbert_indices = generate_sapbert_scores_indices(entity_texts, new_sapbert_index, new_model)

In [ ]:
new_candidate_lookup = { entity_text:new_sapbert_indices[i].tolist() for i,entity_text in enumerate(entity_texts) }
new_scores_lookup = { entity_text:new_sapbert_scores[i].tolist() for i,entity_text in enumerate(entity_texts) }

In [ ]:
new_train_dataset = [
    chunk
    for i, sentence in enumerate(train_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        train_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        new_candidate_lookup,
        new_scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
        force_gold=False
    )
]
    
new_val_dataset = [
    chunk
    for i, sentence in enumerate(val_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        val_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        new_candidate_lookup,
        new_scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

new_test_dataset = [
    chunk
    for i, sentence in enumerate(test_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        test_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        new_candidate_lookup,
        new_scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

In [ ]:
print("New Stage 1 train Hits@5:", hits_at_k_stage1(new_train_dataset, k=5))
print("New Stage 1 val Hits@5:", hits_at_k_stage1(new_val_dataset, k=5))
print("New Stage 1 test Hits@5:", hits_at_k_stage1(new_test_dataset, k=5))